# 部署与监控

来源:
- https://docs.langchain.com/mcp/deploy
- https://docs.langchain.com/mcp/observability
- https://docs.langchain.com/mcp/studio
- https://docs.langchain.com/mcp/ui

本笔记覆盖LangGraph应用的部署、可观测性配置、LangGraph Studio的使用以及REST API/SDK调用。

---
## 1. 部署概述

LangGraph支持多种部署方式，将图(Agent)作为API服务运行。

### 部署架构

```
[客户端] ──HTTP/REST──→ [LangGraph Server] ──→ [LangSmith (监控)]
                           │
                           ├── [Agent 1]
                           ├── [Agent 2]
                           └── [Agent 3]
```

### 部署方式对比

| 方式 | 描述 | 适用场景 |
|------|------|----------|
| **LangGraph Platform** | 全托管部署 | 生产环境、企业级 |
| **自托管 (Docker)** | 自行部署服务器 | 内部系统、私有部署 |
| **Serverless** | 按需运行 | 低频调用、成本优化 |

### LangGraph Server 启动

```python
# langgraph.json 部署配置文件
{
  "dependencies": ["."],
  "graphs": {
    "customer_support": "./src/agent.py:customer_support_graph",
    "research_assistant": "./src/agent.py:research_graph"
  },
  "env": ".env"
}
```

```bash
# 启动本地服务器
langgraph dev
# 部署到LangGraph Platform
langgraph deploy
```

---
## 2. REST API 调用

部署后的Agent可以通过REST API进行同步或流式调用。

### 同步调用

```python
import requests

# 配置
API_URL = "http://localhost:8123"
GRAPH_NAME = "customer_support"

# 同步调用
response = requests.post(
    f"{API_URL}/runs",
    json={
        "assistant_id": GRAPH_NAME,
        "input": {
            "messages": [
                {"role": "human", "content": "我需要查询账单"}
            ]
        }
    }
)
print(response.json())
```

### 流式调用

```python
# 流式响应 (SSE)
import json

response = requests.post(
    f"{API_URL}/runs/stream",
    json={
        "assistant_id": GRAPH_NAME,
        "input": {"messages": [{"role": "human", "content": "你好"}]},
        "stream_mode": "values",
    },
    stream=True
)

for line in response.iter_lines():
    if line:
        data = json.loads(line.decode('utf-8').lstrip('data: '))
        print(data)
```

### Python SDK 调用

```python
from langgraph_sdk import get_client

async def call_agent():
    # 创建客户端
    client = get_client(url="http://localhost:8123")
    
    # 获取助手列表
    assistants = await client.assistants.get()
    
    # 创建运行
    thread = await client.threads.create()
    
    # 同步运行
    result = await client.runs.wait(
        thread_id=thread["thread_id"],
        assistant_id="customer_support",
        input={"messages": [{"role": "human", "content": "你好"}]}
    )
    
    # 流式运行
    async for event in client.runs.stream(
        thread_id=thread["thread_id"],
        assistant_id="customer_support",
        input={"messages": [{"role": "human", "content": "你好"}]}
    ):
        print(event)

import asyncio
asyncio.run(call_agent())
```

### TypeScript SDK 调用

```typescript
// TypeScript
import { Client } from "@langchain/langgraph-sdk";

const client = new Client({
  apiUrl: "http://localhost:8123"
});

// 同步调用
const result = await client.runs.wait(
  "thread-id",
  "customer_support",
  { messages: [{ role: "human", content: "你好" }] }
);

// 流式调用
const stream = client.runs.stream(
  "thread-id",
  "customer_support",
  { messages: [{ role: "human", content: "你好" }] }
);

for await (const event of stream) {
  console.log(event);
}
```

---
## 3. 可观测性 (Observability)

使用LangSmith监控和调试Agent行为。

### 配置LangSmith追踪

```python
import os

# 环境变量配置
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = "lsv2_..."
os.environ["LANGSMITH_PROJECT"] = "my-agent-project"

# 或在代码中配置
from langsmith import Client

ls_client = Client(
    api_url="https://api.smith.langchain.com",
    api_key="lsv2_...",
    project_name="my-agent-project"
)
```

### 可观测数据结构

| 数据 | 说明 | LangSmith位置 |
|------|------|---------------|
| **Run** | 单次图执行 | Traces 页面 |
| **Step** | 图中单个节点执行 | Span 详情 |
| **Token** | LLM调用和消耗 | Monitoring |
| **Latency** | 节点/图执行时间 | Performance |
| **Error** | 异常和错误 | Errors 页面 |

### 自定义追踪

```python
from langsmith import traceable
from langsmith.run_trees import RunTree

# 使用装饰器追踪自定义函数
@traceable(name="custom_processing", run_type="chain")
def process_user_input(text: str) -> str:
    # 自定义逻辑
    return text.strip()

# 创建自定义Run Tree
from langsmith import Client

client = Client()
run_tree = client.create_run(
    name="data_processing",
    run_type="chain",
    inputs={"data": "原始数据"},
    outputs={"result": "处理结果"},
    extra={"version": "1.0"}
)
```

---
## 4. LangGraph Studio

LangGraph Studio 是一个可视化开发和调试工具。

### 启动 Studio

```bash
# 开发模式启动（包含Studio UI）
langgraph dev

# 访问 http://localhost:2024
```

### Studio 功能

| 功能 | 描述 |
|------|------|
| **图可视化** | 实时显示图结构和执行路径 |
| **节点调试** | 查看每个节点的输入/输出 |
| **断点调试** | 在特定节点设置断点 |
| **状态检查** | 查看任意节点的完整状态 |
| **手动干预** | 暂停执行并编辑状态 |
| **回放** | 重新执行特定trace |

### Studio 配置

```python
# 为Studio优化的图配置
from langgraph.checkpoint import MemorySaver

graph = builder.compile(
    checkpointer=MemorySaver(),  # 启用断点恢复
    interrupt_after=["human_review"],  # 在审批节点暂停
    debug=True  # 开发模式
)
```

---
## 5. UI 集成

LangGraph部署的Agent可以通过多种前端UI框架集成。

### API 端点一览

| 端点 | 方法 | 说明 |
|------|------|------|
| `/assistants` | GET | 获取可用助手列表 |
| `/threads` | POST | 创建新对话线程 |
| `/runs` | POST | 同步执行 |
| `/runs/stream` | POST | 流式执行 |
| `/threads/{id}/state` | GET | 获取线程状态 |
| `/threads/{id}/history` | GET | 获取历史消息 |

### 客户端库配置

```python
# Python 客户端
from langgraph_sdk import get_client

client = get_client(url="http://localhost:8123")

# 获取所有助手
assistants = await client.assistants.get()

# 创建新对话
thread = await client.threads.create()

# 发送消息
async for event in client.runs.stream(
    thread["thread_id"],
    "customer_support",
    input={"messages": [{"role": "human", "content": "你好"}]}
):
    # 处理流式事件
    if event.event == "values" and event.data:
        messages = event.data.get("messages", [])
        if messages:
            print(messages[-1].get("content", ""))
```

## 6. 参考链接

- [部署文档](https://docs.langchain.com/mcp/deploy)
- [可观测性文档](https://docs.langchain.com/mcp/observability)
- [LangGraph Studio](https://docs.langchain.com/mcp/studio)
- [UI 集成](https://docs.langchain.com/mcp/ui)
- [LangGraph 部署指南](https://langchain-ai.github.io/langgraph/concepts/deployment/)
- [LangSmith 文档](https://docs.smith.langchain.com/)
- [LangGraph SDK PyPI](https://pypi.org/project/langgraph-sdk/)